# Experiment 1: Baseline

Compare retrieval and generated answers on the same 31 seeded eligible questions and fixed corpus, with a separate IMEI diagnostic.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.

**Reference input scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions and a separate IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** **this notebook Experiment 1 (Baseline)** --> Diagnosis --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## 1. Experiment setup

The table shows the main models and passage budgets. Expand the configuration below it for generation settings, endpoints, retry policies and the system prompt.

`configs/tutorial_inputs.json` declares the frozen inputs; `configs/experiment1.json` defines the scoring settings. The live setup displays the pipeline configuration before model calls.


In [1]:
STUDY = 'baseline'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,32
Generation limit,400 tokens


## 2. Verify the 31 seeded eligible questions

The selected question list, candidates, results and frozen IDs must agree. Inspect the questions below; the code verifies their order and membership.

Question counts and completion counts are checked separately; the baseline averages use only the 31 seeded questions.


In [2]:
if MODE == "plan":
    display(Markdown("Plan: verify the 31 seeded eligible questions and separate IMEI diagnostic."))
else:
    cohort = view.questions()
    baseline = cohort[~cohort.diagnostic]
    assert len(baseline) == 31 and baseline.query_id.is_unique
    assert cohort.loc[cohort.diagnostic, "query_id"].tolist() == ["1053665"]
    display_table(baseline[["query_id", "question"]].reset_index(drop=True))
    display(Markdown("**Verified: 31 eligible baseline questions.** IMEI (1053665) is checked separately and excluded from baseline averages."))


query_id,question
1088186,"weather in texarkana, arkansas"
query_id,question
1038185,what is the ideal credit card balance for credit score
1086238,what causes antibiotic resistance in bacteria
1100277,gestation medical definition
1029544,what is tracers
1082398,what does warbled mean
1055197,what is flex fuel on a jeep?
1089273,types of road hugger tires
1085733,what could cause soreness of heel


**Verified: 31 eligible baseline questions.** IMEI (1053665) is checked separately and excluded from baseline averages.

## 3. Load the questions and fixed corpus

The baseline corpus contains **10,210 deduplicated passages** associated with the **31 seeded baseline questions** and **1,000 additional distractor questions**. Distractor questions were sampled with seed 42 from the same 15,585 eligible IDs, excluding the baseline questions and the original effervescent diagnostic. All associated selected and unselected passages were included; individual distractor passages were not sampled independently and need not be irrelevant to every baseline question.

The dataset revision is pinned. Passages are trimmed, deduplicated by case-insensitive text, assigned stable IDs, and limited to 1,800 characters. The reconstructed corpus must match the fixed corpus.

The IMEI diagnostic **1053665 (what is imei and where do i find it)** is evaluated separately against this unchanged corpus, which contains **0 of its 2 annotated supporting passages**. Its answers and scores remain separate from the 31-question averages. **1100870 (effervescent meaning)** is a different diagnostic and is not relabeled as IMEI.


In [3]:
view.show('corpus')


{'passages': 10210, 'passage_embedding_calls': 0}

id,text,urls,original_char_count,truncated
p_0007d28adb099621fc11,"Recognizing and Rewarding Employee Performance by Kevin Scheid (age) and recognition. We can see from Chart 1 that even though all generations are content with the fairness of leadership, people over 65 years old are significantly more content with the fairness of pay, recognition and rewards. People between the ages of 56 and 65 are consistently more",[http://www.bcwinstitute.com/resources/Press/rp_appreciation.pdf],353,False
p_000f65bcc78f36072154,"Selinker's concept of fossilization is similar to that of Nemser, Tarone, and Sridhar, all of whom attempted to explore the causes of fossilization in second language learners' interlanguage. Fossilization has attracted considerable interest among researchers and has engendered significant differences of opinion.",[https://en.wikipedia.org/wiki/Interlanguage_fossilization],314,False
p_001d29924b5ad5dbc184,"Those people who have bad credit are not being approved by banks and they are finding a higher success rate with car title loans. A bank will most likely turn you down if you have bad credit and request thousands of dollars to consolidate your debt or to pay bills. With an auto title loan, you will get what you need without the hassle. Disadvantages. Car title loans are given on the notion that you will pay them back in a short period of time. The interest rate is low in the beginning, but if you continue to extend your payments further out, the interest rate grows.",[http://www.approvedmoneycenter.com/car-title-loans-blog/car-title-loan/car-title-loans-advantages-and-disadvantages/],572,False


## 4. Document embedding and cache

The baseline reuses existing cached Qwen document embeddings for its **10,210 passages**. The matrix, origin record, and configuration document that reuse; live execution embeds questions without rebuilding document embeddings.

Document vectors are prepared inputs. The query-embedding interface below is used for fresh question embeddings in section 6; document embeddings are not rebuilt.


In [4]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

view.show('document_embeddings')


{'matrix_shape': (10210, 2560),
 'dtype': 'float32',
 'new_document_embedding_calls': 0}

In [5]:
display(Markdown("```python\n" + inspect.getsource(embedding_payload) + "\n```"))


```python
def embedding_payload(model, texts, dimensions, input_type='query', instruction=QUERY_INSTRUCTION):
    if input_type not in {'query', 'passage'}:
        raise ValueError('Embedding mode must be query or passage.')
    inputs = list(texts)
    if input_type == 'query' and instruction:
        inputs = [f'Instruct: {instruction}\nQuery: {text}' for text in inputs]
    return {'model': model, 'input': inputs, 'dimensions': dimensions, 'encoding_format': 'float'}

```

### Cosine retrieval and cache verification

Haystack’s in-memory document store uses the prepared document vectors. Cosine similarity ranks thirty candidates; each RAG path supplies five passages to the generator. Input integrity is checked internally.

In [6]:
# Reuse the prepared document vectors; query embedding happens in section 6.
def retrieve_top30(query_embedding):
    retriever = tutorial.document_retriever()
    return retriever.run(query_embedding=list(query_embedding))["documents"]


## 5. Scoring implementation and definitions

| Metric | definition |
|---|---|
| **Recall@30 / Recall@5** | Unique known support IDs retrieved / all known support IDs |
| Hit@30 / Hit@5 | Any known support ID retrieved |
| MRR@5 | Reciprocal rank of first known support, else 0 |
| nDCG@5 | Binary DCG divided by ideal DCG for min(5, number of gold IDs) |
| Normalized EM | Any reference matches after normalization |
| **Token F1** | Maximum token multiset overlap F1 over references |
| Citation diagnostics | Known gold cited; count of cited IDs absent from selected five |
| Human rubric pass rate | Mean `correct` on questions with both successful arms and both reviews |

Normalization lowercases, removes single/grouped `[p_ID]` citations, removes punctuation and articles (`a`, `an`, `the`), then collapses whitespace. EM and F1 measure lexical agreement, not semantic correctness. Gold labels may be incomplete. Unreviewed human scores remain blank.

In [7]:
from shared_scoring.answers import score_custom_answer, normalize_custom_answer
display(Markdown("```python\n" + inspect.getsource(normalize_custom_answer) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(score_custom_answer) + "\n```"))


```python
def normalize_custom_answer(answer, citation_pattern):
    """Reproduce the preserved custom normalization for cross-checking."""
    text = remove_passage_citations(answer.lower(), citation_pattern)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', text).split())

```

```python
def score_custom_answer(answer, reference_answers, citation_pattern):
    """Return the existing custom exact match and Token F1 on the 0–1 scale."""
    if not reference_answers:
        raise ValueError('Supply all reference answers; an empty reference list is invalid.')
    normalized = normalize_custom_answer(answer, citation_pattern)
    tokens = normalized.split()
    f1 = []
    for reference in reference_answers:
        other = normalize_custom_answer(reference, citation_pattern).split()
        overlap = sum((Counter(tokens) & Counter(other)).values())
        f1.append(2 * overlap / (len(tokens) + len(other)) if tokens or other else 1.0)
    return {'normalized_em': float(any(normalized == normalize_custom_answer(r, citation_pattern) for r in reference_answers)),
            'token_f1': max(f1)}

```

## 6. Retrieval, evidence selection and generation

Follow the question through query embedding, Haystack retrieval of thirty candidates, selection of five passages with or without reranking, and two separate generator requests.

The cells below show the actual Qwen/OpenRouter calls through an OpenAI-compatible client. Live execution performs query embedding, retrieval, reranking and generation for this experiment’s questions.


In [8]:
# Fresh query embeddings and retrieval for 31 questions plus the separate IMEI diagnostic.
retrieval_run = tutorial.retrieve(embed_api, retrieve_api=retrieve_top30)
if retrieval_run is not None:
    display(retrieval_run)
    # The embedding path selects the first five of these thirty candidates.
    view.show("contexts")


retrieve: saved for CURRENT LIVE RUN 20261111T025923Z-420bacd4af50


{'questions': 32,
 'candidate_rows': 960,
 'query_embedding_matrix': 'query_embeddings.npy',
 'candidates': 'retrieved_top30.parquet',
 'passage_embedding_calls': 0,
 'run_id': '20261111T025923Z-420bacd4af50'}

passage_id,rank,is_known_support,text,similarity_score
p_5a45997fb6bc4a13b57a,1,False,"Wireless phone number; Account holder billing address; Phone or device make and model; Date, time, location and detailed description of the incident",0.4231
passage_id,rank,is_known_support,text,similarity_score
p_dd5c85e0ab29e48c2997,2,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.",0.3929
p_998fb287cf11bc761fb7,3,False,Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.,0.3917
p_19cdb207b1973a1a8aa7,4,False,"Create or access your user profile by entering your phone number in Find Your Account below. Once entered, you'll be redirected to your phone provider's program portal. Click on the log on button on the top left menu. A pop up menu will appear with three options: Log On, Sign Up, or Forgot Password. Pick the button that's applicable to you.",0.3871
p_82a8668aab88149e69cf,5,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right corner) to view the Site Code directly below. If you do not have access to your educator account, please contact Customer Care.",0.3866
p_88166e9d68d3c5c8e8e9,6,False,Enter your cell phone number and we will text you {$Name} address and phone number. Phone number ex. 2151234567 or 215-123-4567,0.3809
p_c4c226b47c00de128a55,7,False,"Find a USB port. On laptops, they are typically located on the sides or on the back panel. For desktop computers, most have front ports, as well as several ports on the back side.",0.3758
p_d91ea8a15403bece17ba,8,False,"If you are using BlackBerry Device software version 7.1 or older, on the home screen of your device, click the Options icon. Click Device > About Device Versions. If you are using a BlackBerry 10 device, on the home screen of your device, swipe down from the top of the screen and tap Settings > About.",0.3725
p_77fc23c95b90963fcafa,9,False,Please enter the customer number and access code found on your statement or invoice.,0.3681


In [9]:
def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

reranking_run = tutorial.rerank(rerank_api)
if reranking_run is not None:
    display(reranking_run)
retrieval_metrics_run = tutorial.diagnose()
if retrieval_metrics_run is not None:
    display(retrieval_metrics_run)


rerank: saved for CURRENT LIVE RUN 20261111T025923Z-420bacd4af50


{'questions': 32,
 'candidate_rows': 960,
 'run_id': '20261111T025923Z-420bacd4af50'}

diagnose: saved for CURRENT LIVE RUN 20261111T025923Z-420bacd4af50


{'annotated_questions': 32,
 'opportunities': 3,
 'run_id': '20261111T025923Z-420bacd4af50'}

In [10]:
def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))

generation_run = tutorial.generate(generate_api)
if generation_run is not None:
    display(generation_run)


generate: saved for CURRENT LIVE RUN 20261111T025923Z-420bacd4af50


{'answers': 64, 'questions': 32, 'run_id': '20261111T025923Z-420bacd4af50'}

## 7. Completed requests and failures

A complete pair means two successful answers to the same question, one per RAG path. The baseline contains 31 questions. The IMEI diagnostic is displayed separately later and is excluded from the 31-question averages.

Attempted questions, successful answers, complete pairs and failed requests have different denominators. Failed answers remain unavailable; they do not become zero scores.


In [11]:
view.show('completion')


Measure,Value
run_id,20261111T025923Z-420bacd4af50
questions,32
successful_answers,64
complete_pairs,32
main_complete_pairs,31
truncated_answers,0
request_outcomes,{'ok': 97}


## 8. Per-question scores and answers

Inspect retrieval coverage and answer overlap for each question. The scoring functions are defined in section 5. Questions that fail before retrieval remain explicit failed records with unavailable metrics.

Baseline averages use complete question pairs and exclude the diagnostic.


In [12]:
view.show('retrieval')
view.show('overlap')


query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta,outcome
1029167,what is unifirst corporation,baseline_31,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged
query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta,outcome
1029544,what is tracers,baseline_31,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged
1032146,what is the source of drinking water in virginia,baseline_31,False,2,1.0000,1.0000,1.0000,1.0000,1.0000,0.8772,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged
1033042,current temperature in niagara falls,baseline_31,False,1,1.0000,1.0000,3.0000,1.0000,0.3333,0.5000,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,False,0.0000,unchanged
1038185,what is the ideal credit card balance for credit score,baseline_31,False,1,0.0000,1.0000,10.0000,0.0000,0.0000,0.0000,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,True,0.0000,unchanged
1040180,what is the distance between portland or and forks wa,baseline_31,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged
1050417,what is mutual plan,baseline_31,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged
1053665,what is imei and where do i find it,opportunity,True,2,0.0000,0.0000,Unavailable,0.0000,0.0000,0.0000,0.0000,0.0000,Unavailable,0.0000,0.0000,0.0000,False,0.0000,unchanged
1054544,what is glycyrrhiza glabra? morrow,baseline_31,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False,0.0000,unchanged


query_id,question,arm,diagnostic,subgroup,normalized_em,token_f1
1088186,"weather in texarkana, arkansas",baseline,False,baseline_31,0.0000,0.6667
query_id,question,arm,diagnostic,subgroup,normalized_em,token_f1
1088186,"weather in texarkana, arkansas",reranked,False,baseline_31,0.0000,0.5781
1038185,what is the ideal credit card balance for credit score,baseline,False,baseline_31,0.0000,0.1818
1038185,what is the ideal credit card balance for credit score,reranked,False,baseline_31,0.0000,0.2373
1086238,what causes antibiotic resistance in bacteria,baseline,False,baseline_31,0.0000,0.3091
1086238,what causes antibiotic resistance in bacteria,reranked,False,baseline_31,0.0000,0.3304
1100277,gestation medical definition,baseline,False,baseline_31,0.0000,0.4651
1100277,gestation medical definition,reranked,False,baseline_31,0.0000,0.4583
1029544,what is tracers,baseline,False,baseline_31,0.0000,0.1519


## 9. Baseline results: 31 questions

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

Each automatic comparison uses the same complete baseline pairs. Completion counts and failures are reported separately. Retrieval coverage and answer overlap measure different things.

The new baseline is sampled from eligible labeled questions with seed 42. Support comes from dataset annotations; human review remains pending. The review rubric checks correctness and completeness, support from the supplied evidence, unsupported claims and appropriate uncertainty when evidence is insufficient.

Timing fields are retained here for inspection. **Experiment 4: Latency Tradeoff** measures response time separately.


This **recorded reference run** completed **31 baseline pairs plus the separate IMEI pair**. Mean baseline Token F1 is **38.98 → 36.53%** (embedding → reranked). Judge coverage and failures are reported separately below.

In [13]:
view.show('retrieval_summary')
view.show('overlap_summary')


diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta
0.0000,1.1290,0.9032,1.0000,2.1613,0.9032,0.7231,0.7645,0.8710,1.0000,1.9355,0.8710,0.7742,0.7995,0.0968,-0.0323


outcome,loss,unchanged
,1,30


token_f1,normalized_em
0.3898,0.0000
0.3653,0.0000


### Standard answer evaluation

Evaluation attribution: Hugging Face Evaluate’s SQuAD metric and RAGAS Faithfulness and Answer relevancy.

SQuAD normalized exact match and **Token F1** use **0–100 percentages**; the original answer scorer above uses **0–1**. **Annotated-support Recall@5** remains a separate retrieval metric on **0–1**. **Faithfulness** uses **0–1**; **Answer relevancy** is usually 0–1, with a mathematical cosine range of −1–1. RAGAS judges evidence support and question relevance; it is not human correctness.

Scores below use the same complete baseline pairs and each arm’s exact separate five passages. Truncated answers remain included. Generation and judge outcomes from this Live run remain inspectable. Diagnostic scores remain separate. Live execution computes the standard scores after generation.


In [14]:
# Score only answers produced by this RUN_ID. Live includes RAGAS where declared.
score_run = await tutorial.score()
if score_run is not None:
    display(score_run)


.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1088186 baseline answer_relevancy: ok 0.789920521099257 (12.7s)


1088186 baseline faithfulness: ok 1.0 (15.1s)


1088186 reranked answer_relevancy: ok 0.7718145253013322 (14.7s)


1088186 reranked faithfulness: ok 0.9230769230769231 (25.7s)


1038185 baseline faithfulness: ok 1.0 (11.8s)


... (120 outputs left out)

1053665 baseline faithfulness: ok 1.0 (22.3s)


1053665 reranked answer_relevancy: ok 0.6978628264334471 (9.7s)


1053665 reranked faithfulness: ok 0.08333333333333333 (16.8s)


{'answers': 64,
 'run_id': '20261111T025923Z-420bacd4af50',
 'table': 'scores/per_answer_scores.csv'}

In [15]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(run_ragas_evaluation) + "\n```"))


```python
def score_squad_answers(inputs, metric, citation_pattern, legacy_citation_pattern):
    """Score each saved answer, keeping truncation and the old scorer comparison."""
    records = []
    for row in inputs:
        result = score_squad_answer(metric, row['answer'], row['reference_answers'], row['query_id'], citation_pattern)
        custom = score_custom_answer(row['answer'], row['reference_answers'], legacy_citation_pattern)
        for key, saved in [('token_f1', 'existing_token_f1_0_1'), ('normalized_em', 'existing_normalized_em_0_1')]:
            if saved in row and abs(custom[key] - row[saved]) > 1e-12:
                raise ValueError(f"Existing custom score changed: {row['query_id']} / {row['arm']} / {key}")
        records.append({key: row[key] for key in ['query_id', 'question', 'arm', 'path', 'answer', 'finish_reason', 'truncated',
                                                 'annotated_support_recall_at_5', 'existing_token_f1_0_1', 'existing_normalized_em_0_1']} |
                       {'squad_answer': remove_passage_citations(row['answer'], citation_pattern),
                        'reference_answers': json.dumps(row['reference_answers'], ensure_ascii=False),
                        'squad_exact_match_pct': result['exact_match'], 'squad_token_f1_pct': result['f1'],
                        'token_f1_delta_0_1': result['f1'] / 100 - custom['token_f1'],
                        'normalized_em_delta_0_1': result['exact_match'] / 100 - custom['normalized_em']})
    return pd.DataFrame(records)

```

```python
async def run_ragas_evaluation(inputs, source_directory, run_id, output_directory, settings,
                               allow_model_calls=False, retry_failures=False):
    """Opt in to judge calls; keep successes fixed and archive technical retries."""
    if not allow_model_calls:
        raise PermissionError('Judge calls are disabled. Use load_saved_judge_scores() for replay, or explicitly set allow_model_calls=True.')
    require_model_calls()
    SOURCE, OUT, CFG = Path(source_directory), Path(output_directory), settings
    rows = list(inputs)
    # Reject changed settings before replaying or extending an existing evaluation.
    manifest_path = OUT / 'judge_manifest.json'
    if manifest_path.exists():
        original = read(manifest_path)['settings']
        relevant = lambda values: {key: value for key, value in values.items()
                                  if key.startswith('judge_') or key.startswith('embedding_') or key in
                                  ['api_base', 'enable_thinking', 'relevancy_strictness', 'ragas_response_policy', 'query_instruction']}
        current = relevant(CFG)
        recorded = relevant(original)
        recorded.pop('judge_system_prompt', None)
        current.pop('judge_system_prompt', None)
        if recorded != current:
            raise ValueError('Judge/model settings changed. Use a separate explicit output directory; existing scores stay fixed.')
        for request_path in (OUT / 'judge_requests').glob('*.json'):
            request = read(request_path)
            if request['url'].endswith('/chat/completions'):
                if CFG['judge_system_prompt'] not in request['request_body']['messages'][0]['content']:
                    raise ValueError('Judge system prompt differs from saved requests. Use a separate output directory.')
                break
    existing = { (record['query_id'], record['arm'], record['metric']): record
                for record in [read(path) for path in (OUT / 'ragas_results').glob('*.json')] }
    if all((row['query_id'], row['arm'], metric) in existing and
           (existing[(row['query_id'], row['arm'], metric)]['status'] == 'ok' or
            (not retry_failures and existing[(row['query_id'], row['arm'], metric)]['status'] != 'started'))
           for row in rows for metric in CFG['metrics']):
        return load_saved_judge_scores(OUT, rows, CFG['metrics'], CFG)
    client, metrics = _make_components(SOURCE, run_id, OUT, CFG)
    semaphore=asyncio.Semaphore(CFG['concurrency'])
    async def evaluate(row,name,metric):
        path=OUT/'ragas_results'/f"{row['query_id']}-{row['arm']}-{name}.json";path.parent.mkdir(exist_ok=True)
        fingerprint=evaluation_fingerprint(row, name, CFG, importlib.metadata.version('ragas'))
        if path.exists():
            previous=read(path)
            if previous['fingerprint']!=fingerprint:
                raise ValueError(f'Changed evaluation input/settings: {path.name}. Use a separate output directory.')
            if previous['status']=='ok':return
            if not retry_failures:
                if previous['status']=='started':raise RuntimeError(f'Unfinished metric requires inspection: {path}')
                return
            archive=OUT/'ragas_attempt_history';archive.mkdir(exist_ok=True)
            prior=list(archive.glob(path.stem+'-*.json'))
            if len(prior)>=CFG['metric_max_attempts']-1:return
            if previous['status']=='started':
                previous.update(status='interrupted',score=None,error='Judge pass interrupted to apply rate-limit throttling',ended_utc=now())
            write(archive/f'{path.stem}-{len(prior)+1}.json',previous)
        async with semaphore:
            token=CONTEXT.set({'query_id':row['query_id'],'arm':row['arm'],'metric':name})
            result={'query_id':row['query_id'],'arm':row['arm'],'path':row['path'],'metric':name,'truncated':row['truncated'],
                'fingerprint':fingerprint,'judge_system_prompt':CFG['judge_system_prompt'],'status':'started','started_utc':now(),'context_passage_ids':row['context_passage_ids']}
            save_evaluation_attempt(path,result);start=time.perf_counter()
            try:
                kwargs={'user_input':row['question'],'response':row['answer']}
                if name=='faithfulness':kwargs['retrieved_contexts']=row['retrieved_contexts']
                score=await metric.ascore(**kwargs)
                value=float(score.value)
                if not math.isfinite(value):raise ValueError('RAGAS returned a non-finite score (e.g. no assessable statements)')
                result.update(status='ok',score=value)
            except asyncio.CancelledError:
                result.update(status='interrupted',score=None,error='Explicit judge evaluation interrupted before completion')
                raise
            except ModelBudgetExceeded:
                raise
            except Exception as e:
                result.update(status='error',score=None,error=f'{type(e).__name__}: {e}')
            finally:
                result.update(elapsed_seconds=time.perf_counter()-start,ended_utc=now());save_evaluation_attempt(path,result);CONTEXT.reset(token)
                print(f"{row['query_id']} {row['arm']} {name}: {result['status']} {result.get('score')} ({result['elapsed_seconds']:.1f}s)",flush=True)
    try:await asyncio.gather(*(evaluate(row,name,metric) for row in rows for name,metric in metrics.items()))
    finally:await client.close()
    return load_saved_judge_scores(OUT, rows, CFG['metrics'], CFG)

```

In [16]:
view.show('scores')
view.show('judge_outcomes')


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1088186,"weather in texarkana, arkansas",baseline,baseline,"The weather in Texarkana, Arkansas, on August 30 is expected to be 76 degrees with a chance of patchy rain (83% chance) and winds from the northeast at 13 mph. On August 31, the forecast is 75 degrees with light drizzle (96% chance of rain) and north winds at 9 mph [p_10af63d165c7c485aedb].",stop,False,1.0000,0.6667,0.0000,"The weather in Texarkana, Arkansas, on August 30 is expected to be 76 degrees with a chance of patchy rain (83% chance) and winds from the northeast at 13 mph. On August 31, the forecast is 75 degrees with light drizzle (96% chance of rain) and north winds at 9 mph .","[""The Texarkana forecast for Aug 30 is 76 degrees and Patchy rain possible. There is 83 percentage chance of rain and 13 mph winds from the Northeast. The Texarkana forecast for Aug 31 is 75 degrees and Light drizzle. There is 96 percentage chance of rain and 9 mph winds from the North.""]",0.0000,66.6667,-0.0000,0.0000,1.0000,ok,0.7899,ok,baseline_31,False,52.0000,Nebius,20261111T025923Z-420bacd4af50
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1088186,"weather in texarkana, arkansas",reranked,reranked,"The current weather in Texarkana, Arkansas, includes a forecast for Wednesday, August 30, with a high of 76 degrees, patchy rain possible, an 83% chance of rain, and 13 mph winds from the northeast. Thursday, August 31, is expected to be 75 degrees with light drizzle, a 96% chance of rain, and 9 mph winds from the north [p_10af63d165c7c485aedb]. For reference, Texarkana's climate averages a warmest month in July, with a record high of 114°F in 2007, and a coolest month in December, with a record low of 17°F in 1990 [p_6f23571c07e9942f3b49].",stop,False,1.0000,0.5781,0.0000,"The current weather in Texarkana, Arkansas, includes a forecast for Wednesday, August 30, with a high of 76 degrees, patchy rain possible, an 83% chance of rain, and 13 mph winds from the northeast. Thursday, August 31, is expected to be 75 degrees with light drizzle, a 96% chance of rain, and 9 mph winds from the north . For reference, Texarkana's climate averages a warmest month in July, with a record high of 114°F in 2007, and a coolest month in December, with a record low of 17°F in 1990 .","[""The Texarkana forecast for Aug 30 is 76 degrees and Patchy rain possible. There is 83 percentage chance of rain and 13 mph winds from the Northeast. The Texarkana forecast for Aug 31 is 75 degrees and Light drizzle. There is 96 percentage chance of rain and 9 mph winds from the North.""]",0.0000,57.8125,0.0000,0.0000,0.9231,ok,0.7718,ok,baseline_31,False,92.0000,Nebius,20261111T025923Z-420bacd4af50
1038185,what is the ideal credit card balance for credit score,baseline,baseline,"The ideal credit card balance for a good credit score is one that keeps credit utilization at 20% or less of the credit limit. For example, with a $5,000 limit, the balance should be $1,000 or less [p_cfc2436ca7fad93fec27]. Some credit cards do not report balances to credit bureaus, so using them won't affect your credit score regardless of balance [p_b5fdfb1fff8942366ea0]. However, carrying a balance above the limit negatively impacts credit scores, so staying under the limit is crucial [p_cd8b4ef268dbbc6841aa].",stop,False,0.0000,0.1818,0.0000,"The id

subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
baseline_31,reranked,squad_token_f1_pct,31,38.9759,36.5298,-2.4461,-4.7414,-0.2134
baseline_31,reranked,faithfulness,31,0.9321,0.9373,0.0051,-0.0593,0.0664
baseline_31,reranked,answer_relevancy,31,0.7940,0.8199,0.0259,-0.0112,0.0893
baseline_31,reranked,answer_words,31,54.6774,58.1290,3.4516,-1.2903,8.1298


arm       faithfulness_status
baseline  ok                     32
reranked  ok                     32
Name: outcomes, dtype: int64

arm       answer_relevancy_status
baseline  ok                         32
reranked  ok                         32
Name: outcomes, dtype: int64

### IMEI diagnostic scoring — separate from the 31-question baseline averages

This overview displays the same IMEI records inspected in detail in section 11. Compare both paths for **1053665, “what is imei and where do i find it”**, using the unchanged **10,210-passage corpus**. Neither annotated support is present. The diagnostic’s new answers and deterministic scores are reported separately; it is excluded from this Live run’s 31-question averages.


In [17]:
view.show('walkthrough', contexts=False)


**1053665: what is imei and where do i find it**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The provided passages do not contain information about what an IMEI is or where to find it. [p_5a45997fb6bc4a13b57a], [p_dd5c85e0ab29e48c2997], [p_998fb287cf11bc761fb7], [p_19cdb207b1973a1a8aa7], and [p_82a8668aab88149e69cf] discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.",stop,False,0.0000,0.1538,0.0000,"The provided passages do not contain information about what an IMEI is or where to find it. , , , , and discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,15.3846,-0.0000,0.0000,1.0000,ok,0.0000,ok,opportunity,True,55.0000,Nebius,20261111T025923Z-420bacd4af50
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions [p_d91ea8a15403bece17ba].\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About [p_d91ea8a15403bece17ba].\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.",stop,False,0.0000,0.2752,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions .\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About .\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,27.5229,0.0000,0.0000,0.0833,ok,0.6979,ok,opportunity,True,94.0000,Nebius,20261111T025923Z-420bacd4af50


## 10. Haystack evaluator cross-check

Compare the Haystack recall and MRR with the calculated retrieval scores. The evaluator uses passage IDs and multi-hit recall. Raw exact match is reported separately from normalized exact match.


In [18]:
view.show('haystack_crosscheck')


query_id,arm,recall,mrr
1029167,embedding,1.0000,1.0000
query_id,arm,recall,mrr
1029167,reranker,1.0000,1.0000
1029544,embedding,1.0000,1.0000
1029544,reranker,1.0000,1.0000
1032146,embedding,1.0000,1.0000
1032146,reranker,1.0000,1.0000
1033042,embedding,1.0000,0.3333
1033042,reranker,1.0000,0.5000
1038185,embedding,0.0000,0.0000


## 11. Inspect the separate IMEI diagnostic

This detailed view revisits the same IMEI diagnostic shown in the scoring overview above. Trace **1053665** through the unchanged **10,210-passage baseline corpus**. It contains **0 of 2 annotated supports**. Compare each path’s passages, prompts, answers and diagnostic scores. No human correctness labels are assigned. Later notebooks show the larger-corpus recovery.


In [19]:
view.show('walkthrough')


**1053665: what is imei and where do i find it**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The provided passages do not contain information about what an IMEI is or where to find it. [p_5a45997fb6bc4a13b57a], [p_dd5c85e0ab29e48c2997], [p_998fb287cf11bc761fb7], [p_19cdb207b1973a1a8aa7], and [p_82a8668aab88149e69cf] discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.",stop,False,0.0000,0.1538,0.0000,"The provided passages do not contain information about what an IMEI is or where to find it. , , , , and discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,15.3846,-0.0000,0.0000,1.0000,ok,0.0000,ok,opportunity,True,55.0000,Nebius,20261111T025923Z-420bacd4af50
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions [p_d91ea8a15403bece17ba].\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About [p_d91ea8a15403bece17ba].\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.",stop,False,0.0000,0.2752,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions .\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About .\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,27.5229,0.0000,0.0000,0.0833,ok,0.6979,ok,opportunity,True,94.0000,Nebius,20261111T025923Z-420bacd4af50


### Query embedding → exact baseline top 30

Live execution embeds the question for local Haystack cosine retrieval over the fixed baseline document vectors. No passage embedding request is made. Inspect the thirty candidates below.

In [20]:
view.show('contexts')


passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_5a45997fb6bc4a13b57a,1,9,False,"Wireless phone number; Account holder billing address; Phone or device make and model; Date, time, location and detailed description of the incident",0.4231,0.0000
passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_dd5c85e0ab29e48c2997,2,3,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.",0.3929,0.0000
p_998fb287cf11bc761fb7,3,5,False,Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.,0.3917,0.0000
p_19cdb207b1973a1a8aa7,4,20,False,"Create or access your user profile by entering your phone number in Find Your Account below. Once entered, you'll be redirected to your phone provider's program portal. Click on the log on button on the top left menu. A pop up menu will appear with three options: Log On, Sign Up, or Forgot Password. Pick the button that's applicable to you.",0.3871,0.0000
p_82a8668aab88149e69cf,5,8,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right corner) to view the Site Code directly below. If you do not have access to your educator account, please contact Customer Care.",0.3866,0.0000
p_88166e9d68d3c5c8e8e9,6,16,False,Enter your cell phone number and we will text you {$Name} address and phone number. Phone number ex. 2151234567 or 215-123-4567,0.3809,0.0000
p_c4c226b47c00de128a55,7,13,False,"Find a USB port. On laptops, they are typically located on the sides or on the back panel. For desktop computers, most have front ports, as well as several ports on the back side.",0.3758,0.0000
p_d91ea8a15403bece17ba,8,1,False,"If you are using BlackBerry Device software version 7.1 or older, on the home screen of your device, click the Options icon. Click Device > About Device Versions. If you are using a BlackBerry 10 device, on the home screen of your device, swipe down from the top of the screen and tap Settings > About.",0.3725,0.0009
p_77fc23c95b90963fcafa,9,17,False,Please enter the customer number and access code found on your statement or invoice.,0.3681,0.0000


### IMEI: embedding top five and reranked top five

These are the separate diagnostic’s contexts from the unchanged 10,210-passage corpus. Neither contains an annotated support passage.

In [21]:
# Inspect both selected fives from this run without repeating the reranker call.
if MODE != "plan":
    diagnostic_candidates = view.context_rows("1053665")
    for rank_column in ["rank", "rerank_rank"]:
        display_table(diagnostic_candidates.sort_values(rank_column).head(5))


passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_5a45997fb6bc4a13b57a,1,9,False,"Wireless phone number; Account holder billing address; Phone or device make and model; Date, time, location and detailed description of the incident",0.4231,0.0000
p_dd5c85e0ab29e48c2997,2,3,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.",0.3929,0.0000
p_998fb287cf11bc761fb7,3,5,False,Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.,0.3917,0.0000
p_19cdb207b1973a1a8aa7,4,20,False,"Create or access your user profile by entering your phone number in Find Your Account below. Once entered, you'll be redirected to your phone provider's program portal. Click on the log on button on the top left menu. A pop up menu will appear with three options: Log On, Sign Up, or Forgot Password. Pick the button that's applicable to you.",0.3871,0.0000
p_82a8668aab88149e69cf,5,8,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right corner) to view the Site Code directly below. If you do not have access to your educator account, please contact Customer Care.",0.3866,0.0000


passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_d91ea8a15403bece17ba,8,1,False,"If you are using BlackBerry Device software version 7.1 or older, on the home screen of your device, click the Options icon. Click Device > About Device Versions. If you are using a BlackBerry 10 device, on the home screen of your device, swipe down from the top of the screen and tap Settings > About.",0.3725,0.0009
p_578382512f0fb9935c14,15,2,False,"A BlackBerry is a smartphone manufactured by Research in Motion (RIM). BlackBerries can synchronize email and calendars from enterprise-class email systems, such as Indiana University's Exchange service. Most BlackBerries come with complete alphanumeric keyboards, whereas many smartphones do not.",0.3450,0.0000
p_dd5c85e0ab29e48c2997,2,3,False,"To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.",0.3929,0.0000
p_2e21b43eae835fd0f1cf,18,4,False,"Drivers and Manuals. Identify your Acer Device. Enter your device serial number, SNID or model number",0.3301,0.0000
p_998fb287cf11bc761fb7,3,5,False,Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.,0.3917,0.0000


### IMEI: exact contexts, prompts and two new diagnostic answers

The generator receives only each path’s own five passages. These diagnostic-only answers and scores remain separate from the 31-question averages.

In [22]:
view.show('prompts')
view.show('walkthrough')


**baseline / original**

**reranked / original**

**1053665: what is imei and where do i find it**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The provided passages do not contain information about what an IMEI is or where to find it. [p_5a45997fb6bc4a13b57a], [p_dd5c85e0ab29e48c2997], [p_998fb287cf11bc761fb7], [p_19cdb207b1973a1a8aa7], and [p_82a8668aab88149e69cf] discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.",stop,False,0.0000,0.1538,0.0000,"The provided passages do not contain information about what an IMEI is or where to find it. , , , , and discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,15.3846,-0.0000,0.0000,1.0000,ok,0.0000,ok,opportunity,True,55.0000,Nebius,20261111T025923Z-420bacd4af50
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions [p_d91ea8a15403bece17ba].\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About [p_d91ea8a15403bece17ba].\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.",stop,False,0.0000,0.2752,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions .\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About .\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,27.5229,0.0000,0.0000,0.0833,ok,0.6979,ok,opportunity,True,94.0000,Nebius,20261111T025923Z-420bacd4af50


### IMEI: evaluation of the diagnostic answers

SQuAD measures reference overlap, not evidence support. Annotated-support Recall@5 is zero in both paths because both annotated supports are absent from the corpus. These scores do not enter the 31-question averages.

In [23]:
if MODE != "plan":
    diagnostic_scores = view.scores()
    display_table(diagnostic_scores[diagnostic_scores.query_id.eq("1053665")])


query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The provided passages do not contain information about what an IMEI is or where to find it. [p_5a45997fb6bc4a13b57a], [p_dd5c85e0ab29e48c2997], [p_998fb287cf11bc761fb7], [p_19cdb207b1973a1a8aa7], and [p_82a8668aab88149e69cf] discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.",stop,False,0.0000,0.1538,0.0000,"The provided passages do not contain information about what an IMEI is or where to find it. , , , , and discuss topics such as account information, school Site Codes, phone insurance, and user profiles, but none mention IMEI. Therefore, based on the given passages, the information is insufficient to answer the question.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,15.3846,-0.0000,0.0000,1.0000,ok,0.0000,ok,opportunity,True,55.0000,Nebius,20261111T025923Z-420bacd4af50
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions [p_d91ea8a15403bece17ba].\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About [p_d91ea8a15403bece17ba].\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.",stop,False,0.0000,0.2752,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:\n\n- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions .\n- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About .\n\nAlternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,27.5229,0.0000,0.0000,0.0833,ok,0.6979,ok,opportunity,True,94.0000,Nebius,20261111T025923Z-420bacd4af50


## 12. Validate the run

Validate question IDs, contexts, request coverage, model settings, paired denominators and scoring provenance. Keep the 31 baseline questions and the separate IMEI diagnostic distinct, and verify the frozen input snapshot.

In [24]:
view.show('audit')
tutorial.show()


Check,Status
"Frozen inputs, question IDs and evidence contexts",Verified


**LIVE run `20261111T025923Z-420bacd4af50` — only this run’s answers, scores and timings.**

,query_id,embedding_recall_at_5,embedding_recall_at_30,reranker_recall_at_5
subgroup,,,,
baseline_31,1.019886e+06,0.903226,1.0,0.870968
opportunity,1.053665e+06,0.000000,0.0,0.000000


,query_id,question,subgroup,embedding_recall_at_5,embedding_recall_at_30,run_id,reranker_recall_at_5
31,1053665,what is imei and where do i find it,opportunity,0.0,0.0,20261111T025923Z-420bacd4af50,0.0


,squad_token_f1_pct,faithfulness,answer_relevancy,answer_words
arm,,,,
baseline,38.975851,0.932104,0.793959,54.677419
reranked,36.529789,0.937252,0.819889,58.129032


squad_token_f1_pct  faithfulness  answer_relevancy  \
subgroup    arm                                                            
baseline_31 baseline           38.975851      0.932104          0.793959   
            reranked           36.529789      0.937252          0.819889   

                      answer_words  
subgroup    arm                     
baseline_31 baseline     54.677419  
            reranked     58.129032

... (5 outputs left out)

**Exact context for this answer:**

[p_5a45997fb6bc4a13b57a]
Wireless phone number; Account holder billing address; Phone or device make and model; Date, time, location and detailed description of the incident

[p_dd5c85e0ab29e48c2997]
To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.

[p_998fb287cf11bc761fb7]
Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.

[p_19cdb207b1973a1a8aa7]
Create or access your user profile by entering your phone number in Find Your Account below. Once entered, you'll be redirected to your phone provider's program portal. Click on the log on button on the top left menu. A pop up menu will appear with three options: Log On, Sign Up, or Forgot Password. Pick the button that's applicable to you.

[p_82a8668aab88149e69cf]
To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right corner) to view the Site Code directly below. If you do not have access to your educator account, please contact Customer Care.

#### IMEI — reranked

The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, including smartphones like BlackBerrys. To find the IMEI on a BlackBerry device:

- For BlackBerry Device software version 7.1 or older: Go to the home screen, click the Options icon, then select Device > About Device Versions [p_d91ea8a15403bece17ba].
- For BlackBerry 10 devices: Swipe down from the top of the screen, tap Settings > About [p_d91ea8a15403bece17ba].

Alternatively, the IMEI may also be found on the device's packaging, under the battery (on older models), or by dialing *#06# on the phone.

**Exact context for this answer:**

[p_d91ea8a15403bece17ba]
If you are using BlackBerry Device software version 7.1 or older, on the home screen of your device, click the Options icon. Click Device > About Device Versions. If you are using a BlackBerry 10 device, on the home screen of your device, swipe down from the top of the screen and tap Settings > About.

[p_578382512f0fb9935c14]
A BlackBerry is a smartphone manufactured by Research in Motion (RIM). BlackBerries can synchronize email and calendars from enterprise-class email systems, such as Indiana University's Exchange service. Most BlackBerries come with complete alphanumeric keyboards, whereas many smartphones do not.

[p_dd5c85e0ab29e48c2997]
To find your school's Site Code, please log into your educator account at my.imaginelearning.com and locate your username (upper right... If you can't find the information you need, please click the Chat button, email support@imaginelearning.com, or call or text (866)457-8776.

[p_2e21b43eae835fd0f1cf]
Drivers and Manuals. Identify your Acer Device. Enter your device serial number, SNID or model number

[p_998fb287cf11bc761fb7]
Simply Log On with your mobile phone number and your {{CarrierName}} account PIN or secret answer at Manage My Account. When can I enroll in {{CarrierName}} Phone Insurance? Our enrollment process is quick and painless.